<a href="https://colab.research.google.com/github/jatinkumar-creator/ForensicAI/blob/main/06_evidence_intake_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ForensicAI — Evidence Lifecycle Testing

## Objective
Validate the integrated evidence lifecycle workflow:
1. Evidence processing
2. Evidence registration
3. SHA-256 baseline verification
4. External reference verification
5. Audit history generation
6. Error handling and evidence preservation

All experiments use synthetic evidence files.

In [8]:
from pathlib import Path
import tempfile
import os
import sys

# Set project directory
os.chdir("/content/ForensicAI")
sys.path.insert(0, os.getcwd())

from app.evidence.lifecycle import register_and_verify_evidence
from database.connection import initialize_database

# Create a fresh temporary test directory
test_dir = Path(tempfile.mkdtemp())

# Create synthetic evidence
evidence_path = test_dir / "synthetic_evidence.jpg"
evidence_path.write_bytes(
    bytes.fromhex("FF D8 FF") + b"synthetic evidence data"
)

# Create and initialize test database
db_path = test_dir / "forensicai_test.db"
initialize_database(db_path)

print("Test environment initialized successfully.")
print("Evidence file:", evidence_path)
print("Database:", db_path)
print("Evidence exists:", evidence_path.exists())
print("Database exists:", db_path.exists())

Test environment initialized successfully.
Evidence file: /tmp/tmpssjfxwvl/synthetic_evidence.jpg
Database: /tmp/tmpssjfxwvl/forensicai_test.db
Evidence exists: True
Database exists: True


In [9]:
from pathlib import Path
import hashlib
import tempfile

from app.evidence.lifecycle import register_and_verify_evidence
from database.connection import initialize_database

print("Evidence Lifecycle Testing Environment Ready")

Evidence Lifecycle Testing Environment Ready


In [10]:
from pathlib import Path
import subprocess
import os
import sys

REPO_URL = "https://github.com/jatinkumar-creator/ForensicAI"
REPO_PATH = Path("/content/ForensicAI")

# Clone the repository if it is missing.
if not (REPO_PATH / ".git").exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_PATH)],
        check=True
    )
else:
    # Synchronize the existing repository.
    subprocess.run(
        ["git", "-C", str(REPO_PATH), "pull", "origin", "main"],
        check=True
    )

# Move into the project directory.
os.chdir(REPO_PATH)

# Make the repository available to Python imports.
sys.path.insert(0, str(REPO_PATH))

print("Project directory:", os.getcwd())
print("Repository available:", (REPO_PATH / "app").is_dir())
print("Project setup successful.")

Project directory: /content/ForensicAI
Repository available: True
Project setup successful.


In [11]:
from app.evidence.lifecycle import register_and_verify_evidence
from database.connection import initialize_database

print("Evidence lifecycle import: SUCCESS")
print("Database connection import: SUCCESS")

Evidence lifecycle import: SUCCESS
Database connection import: SUCCESS


In [12]:
#Successful baseline verification
test_dir = Path(tempfile.mkdtemp())

evidence_path = test_dir / "synthetic_evidence.jpg"
evidence_path.write_bytes(
    bytes.fromhex("FF D8 FF") + b"synthetic evidence data"
)

db_path = test_dir / "forensicai_test.db"
initialize_database(db_path)

result = register_and_verify_evidence(
    db_path=db_path,
    source_path=evidence_path
)

print("Lifecycle:", result["lifecycle_status"])
print("Evidence ID:", result["evidence_record"]["evidence_id"])
print("Verification:", result["verification"]["verification_status"])
print("Audit ID:", result["verification"]["audit_id"])

Lifecycle: COMPLETED
Evidence ID: ffd3fc96-1b1b-4e6a-82cf-302302e1c1f6
Verification: MATCH
Audit ID: 9cc7d27c-501c-4e2f-b47e-cf0809316cc6


In [13]:
#External reference mismatch
mismatch_result = register_and_verify_evidence(
    db_path=db_path,
    source_path=evidence_path,
    reference_source="EXAMINER_SUPPLIED",
    expected_sha256="0" * 64,
    reference_description="Deliberately incorrect synthetic reference"
)

print("Lifecycle:", mismatch_result["lifecycle_status"])
print("Verification:", mismatch_result["verification"]["verification_status"])
print("Integrity verified:", mismatch_result["verification"]["integrity_verified"])

Lifecycle: COMPLETED
Verification: MISMATCH
Integrity verified: False


In [14]:
import hashlib

correct_hash = hashlib.sha256(
    evidence_path.read_bytes()
).hexdigest()

match_result = register_and_verify_evidence(
    db_path=db_path,
    source_path=evidence_path,
    reference_source="EXAMINER_SUPPLIED",
    expected_sha256=correct_hash,
    reference_description="Correct synthetic examiner reference",
    actor="Test Examiner"
)

print("Lifecycle:", match_result["lifecycle_status"])
print("Verification:", match_result["verification"]["verification_status"])
print("Integrity verified:", match_result["verification"]["integrity_verified"])
print("Reference source:", match_result["verification"]["reference_source"])

Lifecycle: COMPLETED
Verification: MATCH
Integrity verified: True
Reference source: EXAMINER_SUPPLIED


In [15]:
import sqlite3

connection = sqlite3.connect(db_path)
connection.row_factory = sqlite3.Row

print("=== VERIFICATION HISTORY ===")

verification_rows = connection.execute("""
    SELECT
        evidence_id,
        reference_source,
        verification_status,
        expected_sha256,
        observed_sha256,
        verified_at_utc
    FROM verification_history
    ORDER BY verified_at_utc
""").fetchall()

for row in verification_rows:
    print(dict(row))

print("\n=== AUDIT LOGS ===")

audit_rows = connection.execute("""
    SELECT
        evidence_id,
        action,
        event_timestamp_utc,
        actor,
        details_json
    FROM audit_logs
    ORDER BY event_timestamp_utc
""").fetchall()

for row in audit_rows:
    print(dict(row))

connection.close()

=== VERIFICATION HISTORY ===
{'evidence_id': 'ffd3fc96-1b1b-4e6a-82cf-302302e1c1f6', 'reference_source': 'REGISTERED_BASELINE', 'verification_status': 'MATCH', 'expected_sha256': '596f6b0005a15dc92bb7aeb33fced4caaf707cde94f0a399d48b78fedbe3b5ce', 'observed_sha256': '596f6b0005a15dc92bb7aeb33fced4caaf707cde94f0a399d48b78fedbe3b5ce', 'verified_at_utc': '2026-11-20T07:55:50.397979+00:00'}
{'evidence_id': 'b0e84ed9-58a2-481d-b4fb-c0f3eaf07186', 'reference_source': 'EXAMINER_SUPPLIED', 'verification_status': 'MISMATCH', 'expected_sha256': '0000000000000000000000000000000000000000000000000000000000000000', 'observed_sha256': '596f6b0005a15dc92bb7aeb33fced4caaf707cde94f0a399d48b78fedbe3b5ce', 'verified_at_utc': '2026-11-20T07:55:50.431071+00:00'}
{'evidence_id': 'f2371b73-3969-4e74-a7e4-a02f46c3919c', 'reference_source': 'EXAMINER_SUPPLIED', 'verification_status': 'MATCH', 'expected_sha256': '596f6b0005a15dc92bb7aeb33fced4caaf707cde94f0a399d48b78fedbe3b5ce', 'observed_sha256': '596f6b0005a15d

In [16]:
import hashlib

# Capture the original file before processing.
original_bytes = evidence_path.read_bytes()

original_hash = hashlib.sha256(original_bytes).hexdigest()

# Run another lifecycle operation using synthetic evidence.
integrity_test = register_and_verify_evidence(
    db_path=db_path,
    source_path=evidence_path,
)

# Capture the file after processing.
final_bytes = evidence_path.read_bytes()

final_hash = hashlib.sha256(final_bytes).hexdigest()

print("Original file unchanged:", original_bytes == final_bytes)
print("Original SHA-256:", original_hash)
print("Final SHA-256:", final_hash)
print("Hash unchanged:", original_hash == final_hash)
print("Lifecycle status:", integrity_test["lifecycle_status"])

Original file unchanged: True
Original SHA-256: 596f6b0005a15dc92bb7aeb33fced4caaf707cde94f0a399d48b78fedbe3b5ce
Final SHA-256: 596f6b0005a15dc92bb7aeb33fced4caaf707cde94f0a399d48b78fedbe3b5ce
Hash unchanged: True
Lifecycle status: COMPLETED


In [17]:
%cd /content/ForensicAI
!git pull origin main

/content
From https://github.com/jatinkumar-creator/ForensicAI
 * branch            main       -> FETCH_HEAD
Already up to date.


In [18]:
!python -m pytest tests/unit/test_lifecycle.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 12 items                                                             

tests/unit/test_lifecycle.py::test_lifecycle_registers_and_verifies_baseline PASSED [  8%]
tests/unit/test_lifecycle.py::test_lifecycle_supports_optional_case_id PASSED [ 16%]
tests/unit/test_lifecycle.py::test_lifecycle_supports_acquisition_metadata PASSED [ 25%]
tests/unit/test_lifecycle.py::test_lifecycle_supports_external_reference_match PASSED [ 33%]
tests/unit/test_lifecycle.py::test_lifecycle_records_external_reference_mismatch PASSED [ 41%]
tests/unit/test_lifecycle.py::test_lifecycle_creates_audit_record PASSED [ 50%]
tests/unit/test_lifecycle.py::test_invalid_reference_does_not_reg

In [19]:
!python -m pytest tests/unit/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 77 items                                                             

tests/unit/test_database_connection.py::test_initialize_database_creates_tables PASSED [  1%]
tests/unit/test_database_connection.py::test_initialize_database_creates_parent_directory PASSED [  2%]
tests/unit/test_database_connection.py::test_foreign_keys_are_enabled PASSED [  3%]
tests/unit/test_database_connection.py::test_connection_uses_sqlite_row_factory PASSED [  5%]
tests/unit/test_database_connection.py::test_missing_schema_raises_error PASSED [  6%]
tests/unit/test_database_connection.py::test_empty_schema_raises_error PASSED [  7%]
tests/unit/test_database_connection.py::test_datab

In [20]:
!python -m pytest tests/unit/test_lifecycle.py -v
!python -m pytest tests/unit/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 12 items                                                             

tests/unit/test_lifecycle.py::test_lifecycle_registers_and_verifies_baseline PASSED [  8%]
tests/unit/test_lifecycle.py::test_lifecycle_supports_optional_case_id PASSED [ 16%]
tests/unit/test_lifecycle.py::test_lifecycle_supports_acquisition_metadata PASSED [ 25%]
tests/unit/test_lifecycle.py::test_lifecycle_supports_external_reference_match PASSED [ 33%]
tests/unit/test_lifecycle.py::test_lifecycle_records_external_reference_mismatch PASSED [ 41%]
tests/unit/test_lifecycle.py::test_lifecycle_creates_audit_record PASSED [ 50%]
tests/unit/test_lifecycle.py::test_invalid_reference_does_not_reg

In [22]:
%cd /content/ForensicAI
!git pull origin main
!python -m pytest tests/unit/test_lifecycle.py -v

/content
From https://github.com/jatinkumar-creator/ForensicAI
 * branch            main       -> FETCH_HEAD
Already up to date.
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 12 items                                                             

tests/unit/test_lifecycle.py::test_lifecycle_registers_and_verifies_baseline PASSED [  8%]
tests/unit/test_lifecycle.py::test_lifecycle_supports_optional_case_id PASSED [ 16%]
tests/unit/test_lifecycle.py::test_lifecycle_supports_acquisition_metadata PASSED [ 25%]
tests/unit/test_lifecycle.py::test_lifecycle_supports_external_reference_match PASSED [ 33%]
tests/unit/test_lifecycle.py::test_lifecycle_records_external_reference_mismatch PASSED [ 41%]
tests/unit/test_

In [23]:
%cd /content/ForensicAI
!git pull origin main
!grep -n "^def test_" tests/unit/test_lifecycle.py
!python -m pytest tests/unit/test_lifecycle.py -v

/content
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.50 KiB | 510.00 KiB/s, done.
From https://github.com/jatinkumar-creator/ForensicAI
 * branch            main       -> FETCH_HEAD
   647a6fe..33df376  main       -> origin/main
Updating 647a6fe..33df376
Fast-forward
 app/evidence/lifecycle.py    |  44 ++++++++++++++++
 tests/unit/test_lifecycle.py | 116 +++++++++++++++++++++++++++++++++++++++++++
 2 files changed, 160 insertions(+)
28:def test_lifecycle_registers_and_verifies_baseline(tmp_path):
44:def test_lifecycle_supports_optional_case_id(tmp_path):
57:def test_lifecycle_supports_acquisition_metadata(tmp_path):
80:def test_lifecycle_supports_external_reference_match(tmp_path):
102:def test_lifecycle_records_external_reference_mismatch(tmp_path):
122:def test_lifecycle_creates_audit_record(tmp_

In [24]:
!python -m pytest tests/unit/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 81 items                                                             

tests/unit/test_database_connection.py::test_initialize_database_creates_tables PASSED [  1%]
tests/unit/test_database_connection.py::test_initialize_database_creates_parent_directory PASSED [  2%]
tests/unit/test_database_connection.py::test_foreign_keys_are_enabled PASSED [  3%]
tests/unit/test_database_connection.py::test_connection_uses_sqlite_row_factory PASSED [  4%]
tests/unit/test_database_connection.py::test_missing_schema_raises_error PASSED [  6%]
tests/unit/test_database_connection.py::test_empty_schema_raises_error PASSED [  7%]
tests/unit/test_database_connection.py::test_datab

In [25]:
!python -m pytest tests/unit/test_handling.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 0 items                                                              

============================ no tests ran in 0.01s =============================
ERROR: file or directory not found: tests/unit/test_handling.py



In [26]:
!python -m pytest tests/unit/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 81 items                                                             

tests/unit/test_database_connection.py::test_initialize_database_creates_tables PASSED [  1%]
tests/unit/test_database_connection.py::test_initialize_database_creates_parent_directory PASSED [  2%]
tests/unit/test_database_connection.py::test_foreign_keys_are_enabled PASSED [  3%]
tests/unit/test_database_connection.py::test_connection_uses_sqlite_row_factory PASSED [  4%]
tests/unit/test_database_connection.py::test_missing_schema_raises_error PASSED [  6%]
tests/unit/test_database_connection.py::test_empty_schema_raises_error PASSED [  7%]
tests/unit/test_database_connection.py::test_datab

In [28]:
%cd /content/ForensicAI

!git pull --ff-only origin main

!git log -1 --oneline
!ls -l app/evidence/handling.py
!ls -l tests/unit/test_handling.py

/content
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 10 (delta 6), reused 10 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 4.94 KiB | 1.23 MiB/s, done.
From https://github.com/jatinkumar-creator/ForensicAI
 * branch            main       -> FETCH_HEAD
   33df376..f4a9110  main       -> origin/main
Updating 33df376..f4a9110
Fast-forward
 app/evidence/handling.py    | 273 ++++++++++++++++++++++++++++
 database/schema.sql         |  74 ++++++++
 tests/unit/test_handling.py | 432 ++++++++++++++++++++++++++++++++++++++++++++
 3 files changed, 779 insertions(+)
 create mode 100644 app/evidence/handling.py
 create mode 100644 tests/unit/test_handling.py
f4a9110 (HEAD -> main, origin/main, origin/HEAD) Add append-only evidence handling history
-rw-r--r-- 1 root root 6957 Nov 20 10:20 app/evidence/handling.py
-rw-r--r-- 1 root root 11558 Nov 20 10:20 tests/unit/test_handlin

In [29]:
!python -m pytest tests/unit/test_handling.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 19 items                                                             

tests/unit/test_handling.py::test_record_handling_event_successfully PASSED [  5%]
tests/unit/test_handling.py::test_handling_event_id_is_unique PASSED     [ 10%]
tests/unit/test_handling.py::test_event_type_is_normalized PASSED        [ 15%]
tests/unit/test_handling.py::test_custody_transfer_is_recorded PASSED    [ 21%]
tests/unit/test_handling.py::test_event_timestamp_is_normalized_to_utc PASSED [ 26%]
tests/unit/test_handling.py::test_naive_event_timestamp_is_rejected PASSED [ 31%]
tests/unit/test_handling.py::test_non_utc_event_timestamp_is_rejected PASSED [ 36%]
tests/unit/test_handling

In [30]:
!python -m pytest tests/unit/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 100 items                                                            

tests/unit/test_database_connection.py::test_initialize_database_creates_tables PASSED [  1%]
tests/unit/test_database_connection.py::test_initialize_database_creates_parent_directory PASSED [  2%]
tests/unit/test_database_connection.py::test_foreign_keys_are_enabled PASSED [  3%]
tests/unit/test_database_connection.py::test_connection_uses_sqlite_row_factory PASSED [  4%]
tests/unit/test_database_connection.py::test_missing_schema_raises_error PASSED [  5%]
tests/unit/test_database_connection.py::test_empty_schema_raises_error PASSED [  6%]
tests/unit/test_database_connection.py::test_datab

In [27]:
%cd /content/ForensicAI

!git status -sb
!git log -1 --oneline

!ls -l app/evidence/handling.py
!ls -l tests/unit/test_handling.py

/content
## main...origin/main
33df376 (HEAD -> main, origin/main, origin/HEAD) Add repeated evidence verification workflow and tests
ls: cannot access 'app/evidence/handling.py': No such file or directory
ls: cannot access 'tests/unit/test_handling.py': No such file or directory


In [31]:
%cd /content/ForensicAI
!git pull --ff-only origin main

/content
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 10 (delta 5), reused 10 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 4.61 KiB | 1.15 MiB/s, done.
From https://github.com/jatinkumar-creator/ForensicAI
 * branch            main       -> FETCH_HEAD
   f4a9110..6717786  main       -> origin/main
Updating f4a9110..6717786
Fast-forward
 app/cases/manager.py     | 326 +++++++++++++++++++++++++++++++++++++++++++++++
 database/schema.sql      | 112 ++++++++++++++++
 tests/unit/test_cases.py | 294 ++++++++++++++++++++++++++++++++++++++++++
 3 files changed, 732 insertions(+)
 create mode 100644 app/cases/manager.py
 create mode 100644 tests/unit/test_cases.py


In [32]:
!ls -l app/cases/manager.py
!ls -l tests/unit/test_cases.py

-rw-r--r-- 1 root root 8300 Nov 20 11:11 app/cases/manager.py
-rw-r--r-- 1 root root 7633 Nov 20 11:11 tests/unit/test_cases.py


In [33]:
!python -m pytest tests/unit/test_cases.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 18 items                                                             

tests/unit/test_cases.py::test_cases_table_exists PASSED                 [  5%]
tests/unit/test_cases.py::test_case_events_table_exists PASSED           [ 11%]
tests/unit/test_cases.py::test_case_events_cannot_be_updated PASSED      [ 16%]
tests/unit/test_cases.py::test_case_events_cannot_be_deleted PASSED      [ 22%]
tests/unit/test_cases.py::test_case_rejects_invalid_status PASSED        [ 27%]
tests/unit/test_cases.py::test_create_case_successfully PASSED           [ 33%]
tests/unit/test_cases.py::test_created_case_has_creation_event PASSED    [ 38%]
tests/unit/test_cases.py::test_get_cas

In [34]:
%cd /content/ForensicAI
!python -m pytest tests/unit/ -v

/content
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 118 items                                                            

tests/unit/test_cases.py::test_cases_table_exists PASSED                 [  0%]
tests/unit/test_cases.py::test_case_events_table_exists PASSED           [  1%]
tests/unit/test_cases.py::test_case_events_cannot_be_updated PASSED      [  2%]
tests/unit/test_cases.py::test_case_events_cannot_be_deleted PASSED      [  3%]
tests/unit/test_cases.py::test_case_rejects_invalid_status PASSED        [  4%]
tests/unit/test_cases.py::test_create_case_successfully PASSED           [  5%]
tests/unit/test_cases.py::test_created_case_has_creation_event PASSED    [  5%]
tests/unit/test_cases.py::tes

In [1]:
from pathlib import Path

repo = Path("/content/ForensicAI")
print("Repository exists:", repo.exists())

Repository exists: False


In [2]:
%cd /content/ForensicAI
!git pull origin main

[Errno 2] No such file or directory: '/content/ForensicAI'
/content
fatal: not a git repository (or any of the parent directories): .git


In [3]:
%cd /content
!git clone https://github.com/jatinkumar-creator/ForensicAI.git
%cd /content/ForensicAI

/content
Cloning into 'ForensicAI'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 158 (delta 70), reused 94 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 59.94 KiB | 864.00 KiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/ForensicAI


In [4]:
!ls -l app/evidence/intake.py tests/unit/test_intake.py

-rw-r--r-- 1 root root 1603 Nov 20 16:23 app/evidence/intake.py
-rw-r--r-- 1 root root 2746 Nov 20 16:23 tests/unit/test_intake.py


In [5]:
!python -m pytest tests/unit/test_intake.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 6 items                                                              

tests/unit/test_intake.py::test_intake_evidence_success PASSED           [ 16%]
tests/unit/test_intake.py::test_intake_rejects_unknown_case PASSED       [ 33%]
tests/unit/test_intake.py::test_intake_rejects_closed_case PASSED        [ 50%]
tests/unit/test_intake.py::test_intake_rejects_empty_case_id PASSED      [ 66%]
tests/unit/test_intake.py::test_intake_rejects_non_string_case_id PASSED [ 83%]
tests/unit/test_intake.py::test_intake_rejects_missing_evidence_file PASSED [100%]

============================== 6 passed in 1.48s ===============================


In [6]:
!python -m pytest tests/unit/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ForensicAI
configfile: pyproject.toml
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 124 items                                                            

tests/unit/test_cases.py::test_cases_table_exists PASSED                 [  0%]
tests/unit/test_cases.py::test_case_events_table_exists PASSED           [  1%]
tests/unit/test_cases.py::test_case_events_cannot_be_updated PASSED      [  2%]
tests/unit/test_cases.py::test_case_events_cannot_be_deleted PASSED      [  3%]
tests/unit/test_cases.py::test_case_rejects_invalid_status PASSED        [  4%]
tests/unit/test_cases.py::test_create_case_successfully PASSED           [  4%]
tests/unit/test_cases.py::test_created_case_has_creation_event PASSED    [  5%]
tests/unit/test_cases.py::test_get_cas

Regression testing
## Automated Test Results

Command:
python -m pytest tests/unit/ -v

Expected result at this development checkpoint:
77 passed

The complete regression suite validates the evidence processing,
registry, database, verification, and lifecycle modules.

## Key Observations

- Evidence is processed before registration.
- Registration stores the calculated SHA-256 baseline.
- Verification results are stored in verification history.
- Verification actions generate audit records.
- A mismatch is recorded as a result, not treated as a successful match.
- Original synthetic evidence remains unchanged.
- Registration remains committed if subsequent verification fails.

## Limitations

A calculated baseline hash does not independently prove acquisition authenticity.
A successful hash comparison establishes equality with the supplied reference,
not the authenticity of that reference.